# ResQ-AI Phase 2: Structural Segmentation (Colab) - v2

This notebook provides a complete pipeline to:
1.  **Setup Environment**: Install Kaggle & Ultralytics.
2.  **Download Data**: Fetch **RescueNet** directly from Kaggle (no local download needed).
3.  **Preprocess**: Convert semantic segmentation masks (Grayscale/Indexed) to **YOLOv8 Polygon** format.
4.  **Train**: Fine-tune `yolov8s-seg.pt` on the dataset.
5.  **Export**: Download the trained weights.

In [ ]:
# 1. Setup Environment & API Key
!pip install kaggle ultralytics -q

import os

# Configure Kaggle API Credentials (provided by user)
os.environ['KAGGLE_USERNAME'] = "adityap9116"
os.environ['KAGGLE_KEY'] = "01e6798e4e0457915ccdb14945a83cd1"

print("Environment setup complete.")

In [ ]:
# 2. Download RescueNet Dataset
# Downloads directly to Colab environment
dataset_name = "yaroslavchyrko/rescuenet"
download_path = "/content/rescuenet_raw"

print(f"Downloading {dataset_name}...")
!kaggle datasets download -d {dataset_name} --unzip -p {download_path}

print("Download complete.")

In [ ]:
# 3. Preprocessing: ROBUST Mask -> Polygon Conversion
import cv2
import numpy as np
import glob
from tqdm import tqdm
import shutil

print("--- Starting Robust Conversion ---")

# Output Paths
output_dir = "/content/datasets/rescuenet_yolo"
# Clear old output to be safe
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

for split in ['train', 'val', 'test']:
    os.makedirs(f"{output_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_dir}/labels/{split}", exist_ok=True)

def get_label_path(img_name, label_dir):
    # Case 1: 123_org.jpg -> 123_lab.png
    if '_org' in img_name:
        lbl_name = img_name.replace('_org.jpg', '_lab.png').replace('.jpg', '.png')
    # Case 2: 123.jpg -> 123_lab.png
    else:
        lbl_name = img_name.replace('.jpg', '_lab.png').replace('.png', '_lab.png')
    
    return os.path.join(label_dir, lbl_name)

def mask_to_yolo(mask_path, label_dir, image_name):
    # Read as GRAYSCALE (Value = Class ID + 1 roughly)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: return
    
    h, w = mask.shape[:2]
    output_txt = ""
    
    unique_values = np.unique(mask)
    for val in unique_values:
        if val == 0: continue
        
        # Mapping: val 1 -> class 0, val 2 -> class 1, etc.
        class_id = val - 1 
        
        binary = np.where(mask == val, 255, 0).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for cnt in contours:
            if cv2.contourArea(cnt) < 50: continue
            
            cnt = cnt.reshape(-1, 2)
            normalized_coords = []
            for pt in cnt:
                normalized_coords.append(f"{pt[0]/w:.6f}")
                normalized_coords.append(f"{pt[1]/h:.6f}")
            
            output_txt += f"{class_id} " + " ".join(normalized_coords) + "\n"
            
    if output_txt:
        label_path = os.path.join(label_dir, image_name.replace('.jpg', '.txt').replace('.png', '.txt'))
        with open(label_path, 'w') as f:
            f.write(output_txt)

# Process Splits
download_path = "/content/rescuenet_raw"
img_dir_base = os.path.join(download_path, 'RescueNet')
splits = {'train': 'train', 'test': 'val'}

for src_split, dst_split in splits.items():
    img_dir_src = os.path.join(img_dir_base, src_split, f"{src_split}-org-img")
    lbl_dir_src = os.path.join(img_dir_base, src_split, f"{src_split}-label-img")
    
    if not os.path.exists(img_dir_src): 
        print(f"Skipping {src_split} (folder not found)")
        continue

    images = glob.glob(os.path.join(img_dir_src, '*.*'))
    print(f"Processing {len(images)} images in {src_split}...")
    
    count = 0
    for img_path in tqdm(images, desc=src_split):
        basename = os.path.basename(img_path)
        
        # Check if label exists before copying image
        lbl_path = get_label_path(basename, lbl_dir_src)
        
        if os.path.exists(lbl_path):
            # Copy Image
            shutil.copy(img_path, os.path.join(output_dir, 'images', dst_split, basename))
            # Convert Mask
            mask_to_yolo(lbl_path, os.path.join(output_dir, 'labels', dst_split), basename)
            count += 1
    print(f"Converted {count} pairs from {src_split}")

# Create data.yaml
class_names = [f"Class_{i}" for i in range(25)] # Generous range to avoid errors
yaml_content = f"""
path: {output_dir}
train: images/train
val: images/val
names:
"""
for i, name in enumerate(class_names):
    yaml_content += f"  {i}: {name}\n"

with open(f"{output_dir}/data.yaml", 'w') as f:
    f.write(yaml_content)

print("\nRobust Conversion Complete.")

In [ ]:
# 4. Train YOLOv8 Segmentation
from ultralytics import YOLO
import shutil
import os

# Clear cache to force rescan of new labels
rescan_paths = [
    "/content/datasets/rescuenet_yolo/labels/train.cache",
    "/content/datasets/rescuenet_yolo/labels/val.cache"
]
for p in rescan_paths:
    if os.path.exists(p): os.remove(p)

print("Starting Training...")
model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=f"{output_dir}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='rescuenet_seg',
    device=0
)

In [ ]:
# 5. Download Weights
from google.colab import files

!zip -r best_seg_weights.zip /content/runs/rescuenet_seg/weights/best.pt
files.download('best_seg_weights.zip')